# Unwrapping AI | Zero to Transformer
## Step 08: Decision Trees — Beginner's Learning Book

**Creator:** Nitin K. Sethi

### Learning objectives
- Understand root nodes, decision nodes, branches and leaves.
- Explain Gini impurity and entropy.
- Train and visualise a Decision Tree classifier.
- Evaluate performance and understand overfitting.
- Tune tree complexity using cross-validation.

> Educational exercise using scikit-learn's Breast Cancer Wisconsin dataset. This is not a medical decision tool.

## 1. What is a Decision Tree?

A Decision Tree learns feature-based questions that divide observations into groups.

- **Root node:** first split.
- **Decision node:** a feature-based question.
- **Branch:** outcome of a question.
- **Leaf node:** final prediction.

The algorithm seeks splits that make the resulting groups more homogeneous with respect to the target class.

## 2. Measuring impurity

**Gini impurity**

\\[
Gini = 1 - \\sum_{k=1}^{K} p_k^2
\\]

**Entropy**

\\[
Entropy = -\\sum_{k=1}^{K} p_k \\log_2(p_k)
\\]

Here, \\(p_k\\) is the proportion of class \\(k\\) in a node. A pure node has impurity zero. The tree selects splits that reduce weighted impurity.

## 3. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

RANDOM_STATE = 42

## 4. Load and inspect the dataset

In [ ]:
dataset = load_breast_cancer(as_frame=True)
X = dataset.data
y = dataset.target

print("Feature matrix:", X.shape)
print("Target encoding:", dict(enumerate(dataset.target_names)))
print("Target counts:")
print(y.value_counts())
print("Missing values:", int(X.isna().sum().sum()))
display(X.head())

## 5. Split the data

We use a stratified 80/20 train-test split. The test set is held back for final evaluation; do not repeatedly tune settings against it.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

## 6. Train a baseline Decision Tree

In [ ]:
tree_model = DecisionTreeClassifier(criterion="gini", random_state=RANDOM_STATE)
tree_model.fit(X_train, y_train)

train_pred = tree_model.predict(X_train)
test_pred = tree_model.predict(X_test)

print(f"Training accuracy: {accuracy_score(y_train, train_pred):.3f}")
print(f"Test accuracy:     {accuracy_score(y_test, test_pred):.3f}")
print("Tree depth:", tree_model.get_depth())
print("Number of leaves:", tree_model.get_n_leaves())

### Think about it
If training accuracy is much higher than test accuracy, what might that indicate?

A deep tree can memorise patterns and noise in its training data. This is one reason Decision Trees can overfit.

## 7. Visualise the tree

In [ ]:
plt.figure(figsize=(22, 10))
plot_tree(
    tree_model, feature_names=list(X.columns),
    class_names=list(dataset.target_names), filled=True,
    rounded=True, max_depth=3, fontsize=8
)
plt.title("Decision Tree — first three levels shown")
plt.show()

The plot is limited to three levels for readability. This does not change the depth of the fitted model.

## 8. Evaluate the baseline model

In [ ]:
print(classification_report(y_test, test_pred, target_names=dataset.target_names))

cm = confusion_matrix(y_test, test_pred)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=dataset.target_names).plot(
    cmap="Blues", values_format="d"
)
plt.title("Test-set confusion matrix")
plt.show()

Accuracy is the fraction of correct predictions. Precision, recall and F1-score provide a more detailed view. Choose metrics based on the cost of different errors in the real application.

## 9. Observe how depth affects complexity

In [ ]:
depths = [1, 2, 3, 4, 5, 7, 10, None]
rows = []
for depth in depths:
    model = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE)
    model.fit(X_train, y_train)
    rows.append({
        "max_depth": str(depth),
        "train_accuracy": accuracy_score(y_train, model.predict(X_train)),
        "test_accuracy": accuracy_score(y_test, model.predict(X_test)),
        "leaves": model.get_n_leaves()
    })
display(pd.DataFrame(rows))

This table is for illustration, not hyperparameter selection: repeatedly comparing settings on the test set leaks information from the test set into model development. Use cross-validation on training data for selection.

## 10. Select settings with cross-validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
parameter_grid = {
    "max_depth": [2, 3, 4, 5, 7, None],
    "min_samples_split": [2, 5, 10, 20],
    "criterion": ["gini", "entropy"]
}
search = GridSearchCV(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    parameter_grid, cv=cv, scoring="f1_macro", n_jobs=-1
)
search.fit(X_train, y_train)
print("Best parameters:", search.best_params_)
print("Best mean CV F1:", round(search.best_score_, 3))

## 11. Final evaluation on the untouched test set

In [ ]:
best_tree = search.best_estimator_
final_pred = best_tree.predict(X_test)
print("Final test accuracy:", round(accuracy_score(y_test, final_pred), 3))
print(classification_report(y_test, final_pred, target_names=dataset.target_names))

The test set is evaluated after cross-validation selects settings using training data only. Treat this as the final estimate of performance on unseen data.

## 12. Feature importance

In [ ]:
importance = pd.Series(best_tree.feature_importances_, index=X.columns).sort_values(ascending=False)
display(importance.head(10).to_frame("importance"))
importance.head(10).sort_values().plot(kind="barh", figsize=(8, 5))
plt.title("Top 10 impurity-based feature importances")
plt.xlabel("Importance")
plt.show()

Feature importance describes contribution to impurity reduction in this fitted tree. It does not establish causation.

## 13. Practical exercises

1. Change `criterion` from `gini` to `entropy`. Compare cross-validation scores.
2. Try `min_samples_leaf` values of 1, 2, 5 and 10.
3. Compare shallow and deep trees. What happens to training and validation performance?
4. Explain one root-to-leaf path from the visualisation.
5. Why should the test set not be used for repeated tuning?

### Reflection
Write 3–5 sentences explaining how a Decision Tree predicts and one way to reduce overfitting.

## 14. Key takeaways

- A Decision Tree learns feature-based decision rules.
- Gini impurity and entropy measure class impurity.
- Deep trees can overfit.
- `max_depth`, `min_samples_split` and `min_samples_leaf` control complexity.
- Cross-validation selects settings using training data.
- Keep the test set for final evaluation.

**Next milestone:** Step 09 — Random Forests.

## References
- [Scikit-learn Decision Trees](https://scikit-learn.org/stable/modules/tree.html)
- [DecisionTreeClassifier API](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html)
- [Breast Cancer Wisconsin dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html)